팀원: 3514 최은수, 3204 차마리, 3507 손원희

# 01 거스름돈 문제

In [ ]:
def check_multiplicative(denoms):
    """
    배수 관계 검사:
    큰 화폐가 작은 화폐의 배수인지 확인
    (정렬된 상태 기준)
    """
    for i in range(len(denoms) - 1):
        if denoms[i] % denoms[i + 1] != 0:
            return False
    return True


def greedy_change(denoms, change, stock=None):
    """
    그리디 알고리즘으로 거스름돈 계산
    - stock: None이면 무한, dict이면 유한
    """
    result = {d: 0 for d in denoms}

    for d in denoms:
        if change <= 0:
            break

        if stock is None:
            # 무한 화폐
            cnt = change // d
        else:
            # 유한 화폐
            cnt = min(change // d, stock[d])

        result[d] = cnt
        change -= cnt * d

    if change != 0:
        return None  # 불가능
    return result

def changeF():
  print("=== 1단계: 화폐 단위 입력 ===")
  denoms = list(map(int, input("화폐 단위를 큰 순서대로 입력 (예: 50000 10000 5000 1000): ").split()))
  denoms.sort(reverse=True)

  if not check_multiplicative(denoms):
      print("배수 관계가 아니므로 그리디 알고리즘이 최적해를 보장하지 않습니다.")
      return

  print("✔ 배수 관계 확인 완료")

  print("\n=== 2단계: 화폐 보유 수량 ===")
  mode = input("무한 화폐인가요? (y/n): ").strip().lower()

  stock = None

  if mode == 'n':
      stock = {}
      print("각 화폐별 보유 수량 입력:")
      for d in denoms:
          stock[d] = int(input(f"{d}원: "))

  print("\n=== 3단계: 거래 정보 ===")
  paid = int(input("지불 금액: "))
  price = int(input("구매 금액: "))

  change = paid - price

  if change < 0:
      print("금액이 부족합니다.")
      return

  print(f"거스름돈: {change}원")

  print("\n=== 4단계: 결과 ===")

  result = greedy_change(denoms, change, stock)

  if result is None:
      print("보유 화폐 부족으로 거스름돈 지급 불가")
  else:
      print("거스름돈 지급 결과:")
      for d in denoms:
          if result[d] > 0:
              print(f"{d}원 × {result[d]}개")

In [ ]:
changeF()

# 02 도시 MST 만들고,분할하기

In [15]:
pip install pyvis

1안

In [17]:
# =========================================
# Kruskal MST 시각화
# 흰색 테마 + Color Hunt 팔레트 적용
# =========================================

from pyvis.network import Network

# ---------------------------------
# find 함수 (경로 압축 사용)
# ---------------------------------
def find(parent, x):
    if parent[x] != x:
        parent[x] = find(parent, parent[x])
    return parent[x]


# ---------------------------------
# union 함수
# ---------------------------------
def union(parent, a, b):
    root_a = find(parent, a)
    root_b = find(parent, b)

    if root_a < root_b:
        parent[root_b] = root_a
    else:
        parent[root_a] = root_b


# ---------------------------------
# 현재 집합 상태 출력 함수
# ---------------------------------
def print_sets(parent, N):

    groups = {}

    for i in range(1, N + 1):

        root = find(parent, i)

        if root not in groups:
            groups[root] = []

        groups[root].append(i)

    sorted_groups = []

    for group in groups.values():
        group.sort()
        sorted_groups.append(group)

    sorted_groups.sort(key=lambda x: x[0])

    result = []

    for group in sorted_groups:
        village_str = ", ".join(f"v{v}" for v in group)
        result.append("{" + village_str + "}")

    print(", ".join(result))


# =========================================
# 입력
# =========================================

N, M = map(int, input("정점 수 / 간선 수 입력: ").split())

edges = []

for _ in range(M):
    a, b, cost = map(int, input().split())
    edges.append((cost, a, b))

# 간선 비용 기준 정렬
edges.sort()

# Union-Find 초기화
parent = [i for i in range(N + 1)]

# MST 저장
mst_edges = []

# MST 전체 비용
total_cost = 0

# 초기 상태 출력
print("초기 상태:")
print_sets(parent, N)

# =========================================
# Kruskal 알고리즘 수행
# =========================================

for cost, a, b in edges:

    # 사이클 방지
    if find(parent, a) != find(parent, b):

        union(parent, a, b)

        mst_edges.append((a, b, cost))

        total_cost += cost

        print(f"\n간선 ({a},{b}) 추가 후:")
        print_sets(parent, N)

        # MST 완성
        if len(mst_edges) == N - 1:
            break

# =========================================
# 결과 출력
# =========================================

print("\nMST에 포함된 간선:")

for a, b, cost in mst_edges:
    print(f"({a}, {b}, {cost})")

print(f"\nMST 전체 비용 합: {total_cost}")

# =========================================
# HTML 그래프 생성
# =========================================

net = Network(
    height="900px",
    width="100%",
    bgcolor="#FFFFFF",
    font_color="#000000",
    notebook=False,
)

# 물리 엔진
net.barnes_hut()

# =========================================
# 노드 추가
# =========================================

for node in range(1, N + 1):

    net.add_node(
        node,
        label=f"v{node}",
        color={
            "background": "#8CC0EB",
            "border": "#BFDDF0",
            "highlight": {
                "background": "#FFEBCD",
                "border": "#8CC0EB"
            }
        },
        size=28,
        borderWidth=4,
        font={
            "color": "#000000",
            "size": 18,
            "face": "arial"
        }
    )

# =========================================
# 간선 추가
# =========================================

for cost, a, b in edges:

    # MST 포함 간선
    if (a, b, cost) in mst_edges:

        net.add_edge(
            a,
            b,
            label=str(cost),
            color="#8CC0EB",
            width=5,
            title="MST 간선"
        )

    # 일반 간선
    else:

        net.add_edge(
            a,
            b,
            label=str(cost),
            color="#D9D9D9",
            width=2,
            dashes=True,
            title="선택되지 않은 간선"
        )

# =========================================
# 그래프 옵션
# =========================================

net.set_options("""
const options = {

  "nodes": {
    "shape": "dot",

    "font": {
      "size": 20,
      "face": "arial"
    }
  },

  "edges": {

    "smooth": {
      "type": "dynamic"
    },

    "font": {
      "size": 16,
      "strokeWidth": 0,
      "color": "#000000"
    }
  },

  "physics": {

    "barnesHut": {
      "gravitationalConstant": -2500,
      "springLength": 170,
      "springConstant": 0.04
    },

    "minVelocity": 0.75
  }
}
""")

# =========================================
# HTML 저장
# =========================================

html_file = "kruskal_mst_visualization.html"

net.save_graph(html_file)

# =========================================
# 추가 UI 패널 삽입
# =========================================

with open(html_file, "r", encoding="utf-8") as f:
    html = f.read()

info_panel = f"""
<div style="
position:absolute;
top:20px;
left:20px;
z-index:9999;

background:white;

padding:24px;

border-radius:24px;

border:3px solid #8CC0EB;

width:340px;

font-family:Arial;

box-shadow:0 10px 30px rgba(140,192,235,0.25);
">

<h1 style="
margin-top:0;
color:#8CC0EB;
font-size:30px;
">
Kruskal MST
</h1>

<p style="
color:#BFDDF0;
font-size:16px;
font-weight:bold;
">
Minimum Spanning Tree Visualization
</p>

<hr style="
border:none;
border-top:2px solid #FFEBCD;
margin:20px 0;
">

<h3 style="color:#8CC0EB;">
선택된 간선
</h3>
"""

for a, b, cost in mst_edges:

    info_panel += f"""
    <div style="
    background:#FFF9D2;
    padding:10px;
    border-radius:12px;
    margin-bottom:10px;
    color:#000000;
    font-weight:bold;
    ">
    ({a}, {b}) → 비용 {cost}
    </div>
    """

info_panel += f"""

<hr style="
border:none;
border-top:2px solid #FFEBCD;
margin:20px 0;
">

<h2 style="
color:#8CC0EB;
margin-bottom:0;
">
총 비용
</h2>

<p style="
font-size:32px;
font-weight:bold;
color:#000000;
margin-top:8px;
">
{total_cost}
</p>

</div>
"""

html = html.replace("<body>", f"<body>{info_panel}")

with open(html_file, "w", encoding="utf-8") as f:
    f.write(html)

# =========================================
# 완료 메시지
# =========================================

print("\nHTML 생성 완료")
print(f"{html_file} 파일을 브라우저에서 실행하세요.")

정점 수 / 간선 수 입력: 8 14
1 2 3
1 3 2
1 4 5
2 3 4
2 5 6
3 4 1
3 6 7
4 6 3
4 7 8
5 6 2
5 8 9
6 7 4
6 8 5
7 8 6
초기 상태:
{v1}, {v2}, {v3}, {v4}, {v5}, {v6}, {v7}, {v8}

간선 (3,4) 추가 후:
{v1}, {v2}, {v3, v4}, {v5}, {v6}, {v7}, {v8}

간선 (1,3) 추가 후:
{v1, v3, v4}, {v2}, {v5}, {v6}, {v7}, {v8}

간선 (5,6) 추가 후:
{v1, v3, v4}, {v2}, {v5, v6}, {v7}, {v8}

간선 (1,2) 추가 후:
{v1, v2, v3, v4}, {v5, v6}, {v7}, {v8}

간선 (4,6) 추가 후:
{v1, v2, v3, v4, v5, v6}, {v7}, {v8}

간선 (6,7) 추가 후:
{v1, v2, v3, v4, v5, v6, v7}, {v8}

간선 (6,8) 추가 후:
{v1, v2, v3, v4, v5, v6, v7, v8}

MST에 포함된 간선:
(3, 4, 1)
(1, 3, 2)
(5, 6, 2)
(1, 2, 3)
(4, 6, 3)
(6, 7, 4)
(6, 8, 5)

MST 전체 비용 합: 20

HTML 생성 완료
kruskal_mst_visualization.html 파일을 브라우저에서 실행하세요.


1안 원본

In [9]:
# Kruskal Algorithm + Union-Find
# 최소 신장 트리(MST) 생성

# find 함수 (경로 압축 사용)
def find(parent, x):
    if parent[x] != x:
        parent[x] = find(parent, parent[x])
    return parent[x]


# union 함수
def union(parent, a, b):
    root_a = find(parent, a)
    root_b = find(parent, b)

    if root_a < root_b:
        parent[root_b] = root_a
    else:
        parent[root_a] = root_b


# 현재 집합 상태 출력 함수
def print_sets(parent, N):

    groups = {}

    # 각 정점의 루트 찾기
    for i in range(1, N + 1):
        root = find(parent, i)

        if root not in groups:
            groups[root] = []

        groups[root].append(i)

    # 집합 내부 정렬
    sorted_groups = []

    for group in groups.values():
        group.sort()
        sorted_groups.append(group)

    # 집합 자체 정렬
    sorted_groups.sort(key=lambda x: x[0])

    # 출력 문자열 생성
    result = []

    for group in sorted_groups:
        village_str = ", ".join(f"v{v}" for v in group)
        result.append("{" + village_str + "}")

    print(", ".join(result))


# 입력
N, M = map(int, input().split())

edges = []

for _ in range(M):
    a, b, cost = map(int, input().split())
    edges.append((cost, a, b))

# 간선 비용 기준 정렬
edges.sort()

# Union-Find 초기화
parent = [i for i in range(N + 1)]

# MST 저장
mst_edges = []

# MST 전체 비용
total_cost = 0

# 초기 상태 출력
print("초기 상태:")
print_sets(parent, N)

# Kruskal 알고리즘 수행
for cost, a, b in edges:

    # 사이클이 발생하지 않는 경우만 선택
    if find(parent, a) != find(parent, b):

        union(parent, a, b)

        mst_edges.append((a, b, cost))

        total_cost += cost

        # 간선 추가 후 출력
        print(f"\n간선 ({a},{b}) 추가 후:")
        print_sets(parent, N)

        # MST 완성 시 종료
        if len(mst_edges) == N - 1:
            break

# MST 결과 출력
print("\nMST에 포함된 간선:")

for a, b, cost in mst_edges:
    print(f"({a}, {b}, {cost})")

# 전체 비용 출력
print(f"\nMST 전체 비용 합: {total_cost}")

5 7
1 2 1
1 3 2
2 3 3
2 4 4
3 4 5
3 5 2
4 5 6
초기 상태:
{v1}, {v2}, {v3}, {v4}, {v5}

간선 (1,2) 추가 후:
{v1, v2}, {v3}, {v4}, {v5}

간선 (1,3) 추가 후:
{v1, v2, v3}, {v4}, {v5}

간선 (3,5) 추가 후:
{v1, v2, v3, v5}, {v4}

간선 (2,4) 추가 후:
{v1, v2, v3, v4, v5}

MST에 포함된 간선:
(1, 2, 1)
(1, 3, 2)
(3, 5, 2)
(2, 4, 4)

MST 전체 비용 합: 9


2안

In [6]:
# =========================================
# Kruskal MST + 도시 분할 시각화
# HTML 그래프 UI 자동 생성 버전
# =========================================

from pyvis.network import Network

# -----------------------------
# find 함수 (경로 압축 사용)
# -----------------------------
def find(parent, x):
    if parent[x] != x:
        parent[x] = find(parent, parent[x])
    return parent[x]


# -----------------------------
# union 함수
# -----------------------------
def union(parent, a, b):
    root_a = find(parent, a)
    root_b = find(parent, b)

    if root_a < root_b:
        parent[root_b] = root_a
    else:
        parent[root_a] = root_b


# -----------------------------
# 그룹 반환 함수
# -----------------------------
def get_groups(parent, N):

    groups = {}

    for i in range(1, N + 1):

        root = find(parent, i)

        if root not in groups:
            groups[root] = []

        groups[root].append(i)

    result = []

    for group in groups.values():
        group.sort()
        result.append(group)

    result.sort(key=lambda x: x[0])

    return result


# =========================================
# 입력
# =========================================

N, M = map(int, input("정점 수 / 간선 수 입력: ").split())

edges = []

for _ in range(M):
    a, b, cost = map(int, input().split())
    edges.append((cost, a, b))

# 간선 정렬
edges.sort()

# Union-Find 초기화
parent = [i for i in range(N + 1)]

# MST 저장
mst_edges = []

# 전체 비용
mst_total_cost = 0

# 제거할 최대 간선
max_edge = None

# =========================================
# Kruskal 수행
# =========================================

for cost, a, b in edges:

    if find(parent, a) != find(parent, b):

        union(parent, a, b)

        mst_edges.append((a, b, cost))

        mst_total_cost += cost

        if max_edge is None or cost > max_edge[2]:
            max_edge = (a, b, cost)

        if len(mst_edges) == N - 1:
            break


# =========================================
# 최대 간선 제거 후 그룹 생성
# =========================================

parent2 = [i for i in range(N + 1)]

for a, b, cost in mst_edges:

    if (a, b, cost) != max_edge:
        union(parent2, a, b)

groups = get_groups(parent2, N)

final_cost = mst_total_cost - max_edge[2]

# =========================================
# 그래프 생성
# =========================================

net = Network(
    height="900px",
    width="100%",
    bgcolor="#000000",
    font_color="white",
    notebook=False,
)

# 물리 엔진
net.barnes_hut()

# =========================================
# 노드 색상
# =========================================

group_colors = [
    "#8CC0EB",
    "#BFDDF0",
]

node_group = {}

for idx, group in enumerate(groups):

    for node in group:
        node_group[node] = idx

# =========================================
# 노드 추가
# =========================================

for node in range(1, N + 1):

    color = group_colors[node_group[node] % len(group_colors)]

    net.add_node(
        node,
        label=f"v{node}",
        color=color,
        size=28,
        borderWidth=3,
        font={
            "color": "black",
            "size": 18,
            "face": "arial",
        },
    )

# =========================================
# 전체 간선 추가
# =========================================

for cost, a, b in edges:

    # 제거된 간선
    if (a, b, cost) == max_edge:

        net.add_edge(
            a,
            b,
            label=str(cost),
            color="#FFEBCD",
            width=6,
            dashes=True,
            title="제거된 최대 비용 간선",
        )

    # MST 간선
    elif (a, b, cost) in mst_edges:

        net.add_edge(
            a,
            b,
            label=str(cost),
            color="#8CC0EB",
            width=5,
            title="MST 간선",
        )

    # 일반 간선
    else:

        net.add_edge(
            a,
            b,
            label=str(cost),
            color="#444444",
            width=2,
        )

# =========================================
# 옵션 설정
# =========================================

net.set_options("""
const options = {
  "nodes": {
    "shape": "dot",
    "scaling": {
      "min": 16,
      "max": 32
    }
  },

  "edges": {
    "smooth": {
      "type": "dynamic"
    },

    "font": {
      "size": 16,
      "color": "white",
      "strokeWidth": 0
    }
  },

  "physics": {
    "barnesHut": {
      "gravitationalConstant": -3000,
      "springLength": 180,
      "springConstant": 0.04
    },

    "minVelocity": 0.75
  }
}
""")

# =========================================
# 결과 HTML 생성
# =========================================

html_file = "kruskal_city_division.html"

net.save_graph(html_file)

# =========================================
# HTML 추가 커스터마이징
# =========================================

with open(html_file, "r", encoding="utf-8") as f:
    html = f.read()

info_panel = f"""
<div style="
position:absolute;
top:20px;
left:20px;
z-index:9999;
background:#111111;
padding:20px;
border-radius:20px;
border:2px solid #8CC0EB;
width:320px;
font-family:Arial;
color:white;
box-shadow:0 0 20px rgba(140,192,235,0.3);
">

<h2 style="margin-top:0;color:#8CC0EB;">
Kruskal MST
</h2>

<p style="color:#FFF9D2;">
도시를 두 구역으로 분할
</p>

<hr style="border-color:#333;">

<p>
<b style="color:#8CC0EB;">총 MST 비용:</b>
{mst_total_cost}
</p>

<p>
<b style="color:#FFEBCD;">제거된 간선:</b><br>
({max_edge[0]}, {max_edge[1]}) / 비용 {max_edge[2]}
</p>

<p>
<b style="color:#BFDDF0;">최종 유지 비용:</b>
{final_cost}
</p>

<hr style="border-color:#333;">

<p><b>구역 정보</b></p>
"""

for i, group in enumerate(groups):

    villages = ", ".join(f"v{v}" for v in group)

    info_panel += f"""
    <div style="
    background:{group_colors[i]};
    color:black;
    padding:10px;
    border-radius:12px;
    margin-bottom:10px;
    font-weight:bold;
    ">
    구역 {i+1}<br>
    {villages}
    </div>
    """

info_panel += "</div>"

html = html.replace("<body>", f"<body>{info_panel}")

with open(html_file, "w", encoding="utf-8") as f:
    f.write(html)

# =========================================
# 출력
# =========================================

print("\nHTML 생성 완료")
print(f"{html_file} 파일을 브라우저로 실행하세요.")

정점 수 / 간선 수 입력: 5 7
1 2 1
1 3 2
2 3 3
2 4 4
3 4 5
3 5 2
4 5 6

HTML 생성 완료
kruskal_city_division.html 파일을 브라우저로 실행하세요.


2안 원본

In [ ]:
# Kruskal Algorithm + Union-Find
# 도시를 두 구역으로 분할하기

# find 함수 (경로 압축 사용)
def find(parent, x):
    if parent[x] != x:
        parent[x] = find(parent, parent[x])
    return parent[x]


# union 함수
def union(parent, a, b):
    root_a = find(parent, a)
    root_b = find(parent, b)

    if root_a < root_b:
        parent[root_b] = root_a
    else:
        parent[root_a] = root_b


# 현재 집합 상태 출력 함수
def print_sets(parent, N):

    groups = {}

    # 그룹 생성
    for i in range(1, N + 1):
        root = find(parent, i)

        if root not in groups:
            groups[root] = []

        groups[root].append(i)

    # 그룹 정렬
    sorted_groups = []

    for group in groups.values():
        group.sort()
        sorted_groups.append(group)

    sorted_groups.sort(key=lambda x: x[0])

    # 출력 문자열 생성
    result = []

    for group in sorted_groups:
        village_str = ", ".join(f"v{v}" for v in group)
        result.append("{" + village_str + "}")

    print(", ".join(result))


# 그룹 반환 함수
def get_groups(parent, N):

    groups = {}

    for i in range(1, N + 1):
        root = find(parent, i)

        if root not in groups:
            groups[root] = []

        groups[root].append(i)

    result = []

    for group in groups.values():
        group.sort()
        result.append(group)

    result.sort(key=lambda x: x[0])

    return result


# 입력
N, M = map(int, input().split())

edges = []

for _ in range(M):
    a, b, cost = map(int, input().split())
    edges.append((cost, a, b))

# 간선 비용 기준 정렬
edges.sort()

# Union-Find 초기화
parent = [i for i in range(N + 1)]

# MST 저장
mst_edges = []

# 전체 MST 비용
mst_total_cost = 0

# 가장 큰 비용 간선
max_edge = None

# 초기 상태 출력
print("초기 상태:")
print_sets(parent, N)

# Kruskal 알고리즘 수행
for cost, a, b in edges:

    # 사이클이 생기지 않는 경우만 선택
    if find(parent, a) != find(parent, b):

        union(parent, a, b)

        mst_edges.append((a, b, cost))

        mst_total_cost += cost

        # 최대 비용 간선 갱신
        if max_edge is None or cost > max_edge[2]:
            max_edge = (a, b, cost)

        # 집합 상태 출력
        print(f"\n간선 ({a},{b}) 추가 후:")
        print_sets(parent, N)

        # MST 완성
        if len(mst_edges) == N - 1:
            break

# 제거할 간선 출력
print("\n제거한 간선:")
print(f"({max_edge[0]}, {max_edge[1]}, {max_edge[2]})")

# 제거 후 새로운 Union-Find 생성
parent2 = [i for i in range(N + 1)]

# 제거한 간선을 제외하고 다시 union 수행
for a, b, cost in mst_edges:

    if (a, b, cost) != max_edge:
        union(parent2, a, b)

# 그룹 정보 가져오기
groups = get_groups(parent2, N)

# 각 구역 비용 계산
region_costs = []

for group in groups:

    cost_sum = 0

    group_set = set(group)

    for a, b, cost in mst_edges:

        if (a, b, cost) != max_edge:

            if a in group_set and b in group_set:
                cost_sum += cost

    region_costs.append(cost_sum)

# 각 구역 출력
for i in range(len(groups)):

    villages = ", ".join(f"v{v}" for v in groups[i])

    print(f"\n구역 {i + 1}:")
    print(f"마을 목록: {{{villages}}}")
    print(f"유지비용 합: {region_costs[i]}")

# 전체 유지비용 합 출력
final_cost = mst_total_cost - max_edge[2]

print(f"\n두 구역 유지비용 총합: {final_cost}")

5 7 
1 2 1
1 3 2
2 3 3
2 4 4
3 4 5
3 5 2
4 5 6
{v1}
{v2}, {v3}, {v4}, {v5}

간선 (3,5) 추가 후:
{v2}, {v3, v5}, {v4}

간선 (2,3) 추가 후:
{v2, v3, v5}, {v4}

간선 (2,4) 추가 후:
{v2, v3, v4, v5}
{v1}, {v2}

간선 (1,2) 추가 후:
{v1, v2}
{v3}, {v4}, {v5}

간선 (3,5) 추가 후:
{v3, v5}, {v4}

간선 (3,4) 추가 후:
{v3, v4, v5}
{v1}, {v3}

간선 (1,3) 추가 후:
{v1, v3}
{v2}, {v4}, {v5}

간선 (2,4) 추가 후:
{v2, v4}, {v5}

간선 (4,5) 추가 후:
{v2, v4, v5}
{v1}, {v4}
{v1}, {v5}

최종 최소 비용 분할 결과

구역 1:
마을 목록: {v1, v2}
유지비용 합: 1

MST 간선:
(1, 2, 1)

구역 2:
마을 목록: {v3, v4, v5}
유지비용 합: 7

MST 간선:
(3, 5, 2)
(3, 4, 5)

두 구역 유지비용 총합: 8


# 03 Prim 알고리즘

1. Prim 알고리즘을 소개하고, 동작 과정을 제시한 후, 아래 그래프에 적용해 단계별로 설명하시오.

    Prim Algorithm은 가중치가 있는 무방향 그래프에서 MST를 구성하기 위한 대표적인 탐욕 알고리즘이다. 최소 신장 트리는 그래프의 모든 정점을 포함하면서도 사이클을 포함하지 않고, 전체 간선 가중치의 합이 최소가 되는 트리를 의미한다.

    Prim 알고리즘은 정점 중심의 접근 방식을 사용한다. 먼저 임의의 시작 정점을 하나 선택한 뒤, 해당 정점을 포함한 집합을 점진적으로 확장해 나간다. 알고리즘의 핵심은 현재까지 선택된 정점 집합과 아직 선택되지 않은 정점 집합을 연결하는 간선들 중에서 가중치가 가장 작은 간선을 선택하여 새로운 정점을 추가하는 것이다. 이 과정을 반복하면 결국 모든 정점이 하나의 트리 구조로 연결되며, 그 결과 최소 신장 트리가 완성된다.

    동작 과정은 먼저 그래프에서 시작 정점을 하나 선택하고, 이를 방문 집합에 포함한다. 이후 방문된 정점들과 방문되지 않은 정점들 사이에 존재하는 모든 간선 중에서 가중치가 최소인 간선을 선택한다. 이 간선이 연결하는 새로운 정점을 방문 집합에 추가하고, 같은 방식으로 선택 가능한 간선들을 계속 갱신한다. 이러한 과정을 모든 정점이 방문될 때까지 반복한다.

    이 과정을 그래프에 적용하면 다음과 같은 방식으로 진행된다. 초기에는 하나의 정점만이 선택된 상태에서 시작하며, 해당 정점에 인접한 간선들이 후보가 된다. 이후 단계에서는 선택된 정점 집합에서 외부로 연결되는 간선들 중 최소 가중치 간선을 선택하여 트리를 확장한다. 이때 중요한 점은 항상 “현재 트리와 외부 정점을 연결하는 가장 작은 비용의 간선”을 선택한다는 것이다. 이러한 선택 방식은 지역적으로 최적의 선택이 전체적으로도 최적해를 보장하는 구조를 만들기 때문에 탐욕 알고리즘으로 분류된다. 최종적으로 모든 정점이 연결되면 사이클이 없는 최소 비용의 트리가 완성된다.

    주어진 그래프에서 정점은 1부터 5까지이며, 시작 정점은 2로 설정한다. Prim Algorithm은 현재 선택된 정점 집합과 외부 정점 집합 사이를 연결하는 간선 중 가중치가 가장 작은 간선을 반복적으로 선택하여 최소 신장 트리를 구성하는 알고리즘이다.

    먼저 시작 정점 2만이 트리에 포함된 상태에서 알고리즘이 시작된다. 정점 2와 연결된 간선은 (2–1, 가중치 1), (2–3, 가중치 3), (2–4, 가중치 6)이다. 이 중 가장 작은 가중치를 가지는 간선은 2–1이므로 해당 간선을 선택하고 정점 1을 트리에 추가한다. 현재까지 선택된 간선은 2–1이며 트리의 정점 집합은 {2, 1}이 된다.

    다음 단계에서는 정점 집합 {2, 1}에서 외부 정점으로 연결되는 모든 간선을 고려한다. 후보 간선은 1–3(3), 2–3(3), 2–4(6)이다. 이 중 가장 작은 가중치를 가지는 간선은 1–3이므로 이를 선택하여 정점 3을 추가한다. 이 시점에서 정점 집합은 {2, 1, 3}이 된다.

    세 번째 단계에서는 현재 집합 {2, 1, 3}에서 외부 정점으로 연결되는 간선을 확인한다. 후보는 3–5(2), 3–4(4), 2–4(6)이며, 가장 작은 가중치는 3–5이므로 이를 선택하여 정점 5를 트리에 포함시킨다. 이제 정점 집합은 {2, 1, 3, 5}가 된다.

    마지막으로 남은 정점 4를 포함하기 위해 가능한 간선은 3–4(4), 5–4(5), 2–4(6)이다. 이 중 최소 가중치는 3–4이므로 해당 간선을 선택하고 정점 4를 추가한다. 이로써 모든 정점이 트리에 포함되어 최소 신장 트리가 완성된다.

    최종적으로 선택된 간선은 2–1, 1–3, 3–5, 3–4이며 전체 비용은 10이 된다. 이 과정은 항상 현재 트리와 외부 정점을 연결하는 최소 가중치 간선을 선택하는 greedy 전략을 기반으로 하며, 이를 통해 전체 그래프에 대해 최소 비용의 연결 구조가 보장된다.

2. 코드 구현

In [ ]:
import heapq
from collections import defaultdict

class Graph:
    def __init__(self, vertices):
        self.V = vertices
        self.graph = defaultdict(list)

    def add_edge(self, u, v, w):
        # 무방향 그래프
        self.graph[u].append((w, v))
        self.graph[v].append((w, u))

    def prim_mst(self, start=0):
        visited = set()
        min_heap = [(0, start, -1)]  # (weight, current_node, parent_node)
        mst_edges = []
        total_cost = 0

        while min_heap and len(visited) < self.V:
            weight, u, parent = heapq.heappop(min_heap)

            if u in visited:
                continue

            visited.add(u)
            total_cost += weight
            if parent != -1:
                mst_edges.append((parent, u, weight))

            for w, v in self.graph[u]:
                if v not in visited:
                    heapq.heappush(min_heap, (w, v, u))

        print("MST 간선 목록:")
        for u, v, w in mst_edges:
            print(f"{u} - {v} (weight={w})")

        print("\n총 비용:", total_cost)

In [ ]:
g = Graph(5)

g.add_edge(1, 2, 1)
g.add_edge(1, 3, 3)
g.add_edge(2, 3, 3)
g.add_edge(2, 4, 6)
g.add_edge(3, 4, 4)
g.add_edge(3, 5, 2)
g.add_edge(4, 5, 5)

g.prim_mst(2)

MST 간선 목록:
2 - 1 (weight=1)
1 - 3 (weight=3)
3 - 5 (weight=2)
3 - 4 (weight=4)

총 비용: 10


3. Kruskal vs Prim

    |항목|Prim|Kruskal|
    |:--:|:--:|:--:|
    |전략|정점 기반|간선 기반|
    |시작|임의 정점|전체 간선 정렬|
    |자료구조|우선순위 큐|Union-Find|
    |동작 방식|정점을 하나씩 추가하며 트리를 확장|간선을 하나씩 선택하며 합침|
    |정렬|필요 없음(큐에서 추출)|전체 간선 사전 정렬 필수|
    |사이클 확인 방식|방문 여부로 확인|Union-Find 자료구조로 확인|
    |그래프 특성|밀집 그래프에 유리|희소 그래프에 유리|


    프림(Prim) 알고리즘은 정점 중심의 탐욕 알고리즘으로, 시작 정점에서 출발하여 현재 구성된 트리와 외부 정점을 연결하는 간선 중 가중치가 가장 작은 간선을 선택하며 점진적으로 최소 신장 트리를 확장한다. 이 과정에서 우선순위 큐를 활용하여 후보 간선들 중 최소 가중치를 효율적으로 관리하며, 특히 간선이 많이 존재하는 밀집 그래프에서 높은 효율을 보인다.

    반면, 크루스칼(Kruskal) 알고리즘은 간선 중심의 접근 방식을 사용한다. 모든 간선을 가중치 기준으로 정렬한 뒤, 비용이 낮은 간선부터 순차적으로 선택하여 트리에 추가한다. 이때 사이클 발생 여부를 판단하기 위해 Union-Find 자료구조를 사용한다. 크루스칼 알고리즘은 간선 수가 상대적으로 적은 희소 그래프에서 정렬 비용 대비 효율이 높아 성능이 우수하다.

# 04 허프만코딩

In [ ]:
import heapq
from collections import Counter
import sys

# 재귀 깊이 제한 증가 (매우 긴 입력 대비)
sys.setrecursionlimit(10000)


# 허프만 트리의 각 노드를 표현하는 클래스
class Node:

    # 노드 생성 순서 저장용
    count = 0

    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

        # 동일 빈도수일 때 안정적인 비교를 위한 순서값
        self.order = Node.count
        Node.count += 1

    # 힙(Heap)에서 우선순위 비교
    def __lt__(self, other):

        # 빈도수가 같으면 생성 순서 비교
        if self.freq == other.freq:
            return self.order < other.order

        return self.freq < other.freq


# 허프만 트리 생성
def build_huffman_tree(text):

    # 빈 문자열 예외 처리
    if not text:
        return None

    # 문자 빈도수 계산
    frequency = Counter(text)

    # 우선순위 큐(최소 힙) 생성
    heap = [Node(char, freq) for char, freq in frequency.items()]
    heapq.heapify(heap)

    # 노드가 하나 남을 때까지 반복
    while len(heap) > 1:

        # 가장 빈도수가 작은 두 노드 선택
        left_node = heapq.heappop(heap)
        right_node = heapq.heappop(heap)

        # 새로운 부모 노드 생성
        merged = Node(None, left_node.freq + right_node.freq)
        merged.left = left_node
        merged.right = right_node

        # 힙에 다시 삽입
        heapq.heappush(heap, merged)

    return heap[0]


# 허프만 코드 생성
def generate_codes(node, current_code="", codes=None):

    if codes is None:
        codes = {}

    # 노드가 없으면 종료
    if node is None:
        return codes

    # 리프 노드인 경우
    if node.char is not None:

        # 문자 종류가 하나뿐인 경우 처리
        codes[node.char] = current_code if current_code != "" else "0"

        return codes

    # 왼쪽은 0
    generate_codes(node.left, current_code + "0", codes)

    # 오른쪽은 1
    generate_codes(node.right, current_code + "1", codes)

    return codes


# 문자열 압축
def encode(text, codes):

    try:
        # 각 문자를 허프만 코드로 변환
        return ''.join(codes[char] for char in text)

    except KeyError:
        raise ValueError("허프만 코드에 존재하지 않는 문자가 포함되어 있습니다.")


# 문자열 복원
def decode(encoded_text, root):

    # 트리가 없는 경우
    if root is None:
        return ""

    # 문자 종류가 하나뿐인 경우
    if root.char is not None:
        return root.char * len(encoded_text)

    decoded_output = []
    current_node = root

    # 비트열 순회
    for bit in encoded_text:

        # 잘못된 비트값 검사
        if bit not in ('0', '1'):
            raise ValueError("압축 데이터에 잘못된 비트값이 포함되어 있습니다.")

        # 트리 이동
        if bit == '0':
            current_node = current_node.left
        else:
            current_node = current_node.right

        # 손상된 데이터 검사
        if current_node is None:
            raise ValueError("손상된 압축 데이터입니다.")

        # 리프 노드 도달
        if current_node.char is not None:
            decoded_output.append(current_node.char)

            # 다시 루트로 복귀
            current_node = root

    # 비트열이 중간에서 끊긴 경우 검사
    if current_node != root:
        raise ValueError("압축 데이터가 중간에서 끊어졌습니다.")

    return ''.join(decoded_output)


# ---------------- 메인 실행 ----------------

if __name__ == "__main__":

    try:
        # 문자열 입력
        text = input("문자열 입력: ")

        # 빈 문자열 처리
        if text == "":
            print("\n[오류] 빈 문자열입니다.")
            sys.exit()

        # 공백만 입력된 경우 처리
        if text.strip() == "":
            print("\n[오류] 공백만 입력되었습니다.")
            sys.exit()

        # 너무 긴 문자열 처리
        MAX_LENGTH = 10000

        if len(text) > MAX_LENGTH:
            print("\n[오류] 문자열 길이가 너무 깁니다.")
            sys.exit()

        print(f"\n원본 문자열: {text}")

        # 허프만 트리 생성
        root = build_huffman_tree(text)

        # 허프만 코드 생성
        huffman_codes = generate_codes(root)

        print("\n[생성된 허프만 코드]")

        for char, code in sorted(huffman_codes.items()):

            # 공백 문자 출력
            if char == ' ':
                print(f"'공백' : {code}")

            # 줄바꿈 문자 출력
            elif char == '\n':
                print(f"'\\n' : {code}")

            # 탭 문자 출력
            elif char == '\t':
                print(f"'\\t' : {code}")

            else:
                print(f"'{char}' : {code}")

        # 문자열 압축
        encoded_text = encode(text, huffman_codes)

        print("\n[압축된 비트열]")
        print(encoded_text)

        # 압축률 계산 (UTF-8 기준)
        original_bits = len(text.encode('utf-8')) * 8
        compressed_bits = len(encoded_text)

        compression_rate = (
            (1 - compressed_bits / original_bits) * 100
        )

        print(f"\n압축 전 비트 수 : {original_bits}")
        print(f"압축 후 비트 수 : {compressed_bits}")
        print(f"압축률 : {compression_rate:.2f}% 절약")

        # 문자열 복원
        decoded_text = decode(encoded_text, root)

        print("\n[복원된 문자열]")
        print(decoded_text)

        # 복원 검증
        if decoded_text == text:
            print("\n복원 성공!")
        else:
            print("\n복원 실패!")

    except ValueError as e:
        print(f"\n[오류 발생] {e}")

    except Exception as e:
        print(f"\n[예상하지 못한 오류 발생] {e}")

문자열 입력: huffman coding

원본 문자열: huffman coding

[생성된 허프만 코드]
'공백' : 1100
'a' : 1011
'c' : 1101
'd' : 1111
'f' : 010
'g' : 001
'h' : 1000
'i' : 000
'm' : 1010
'n' : 011
'o' : 1110
'u' : 1001

[압축된 비트열]
10001001010010101010110111100110111101111000011001

압축 전 비트 수 : 112
압축 후 비트 수 : 50
압축률 : 55.36% 절약

[복원된 문자열]
huffman coding

복원 성공!


1. 해 선택 ( greedy choice )
: 현재 남아 있는 노드들 중에서 빈도수가 가장 작은 두 노드를 선택한다. 허프만 코딩에서는 빈도수가 작은 문자가 긴 코드를 가지도록 하는 것이 전체 압축 효율을 높이므로, 가장 작은 두 노드를 우선적으로 합치는 것이 가장 좋은 선택이 된다. 프로그램에서는 최소 힙 ( 우선순위 큐 ) 을 이용하여 수행한다.
left_node = heapq.heappop(heap)
right_node = heapq.heappop(heap)
매 단계마다 현재 가장 비용이 작은 두 노드를 선택하는 탐욕적 선택을 수행한다.

2. 실행 가능성 검사 ( feasibility check )
: 선택한 두 노드를 하나의 부모 노드로 합쳐도 허프만 트리의 조건을 만족하는지 확인한다. 부모 노드의 빈도수가 두 자식 노드 빈도수의 합인지, 기존 트리 구조가 유지되는지, 모든 문자가 리프 노드에 위치하도록 유지되는지 확인한다. 새 부모 노드 생성은
merged = Node(None, left_node.freq + right_node.freq)
merged.left = left_node
merged.right = right_node
를 이용한다. 이 과정에서 허프만 트리의 구조 조건을 만족하는 새로운 부분 해를 만든다.

3. 해 검사 ( solution check )
: 허프만 트리가 완성되었는지 확인한다. 모든 노드가 하나의 트리로 합쳐져 힙에 노드가 하나만 남으면 최종 허프만 트리가 완성된 것이다. 프로그램에서는 다음 반복 조건으로 확인한다.
while len(heap) > 1:
return heap[0]
를 통해 최종 루트 노드를 반환한다. 즉, 힙에 노드가 하나만 남은 상태가 전체 문제의 최적해가 완성된 상태이다.

허프만 코딩은 매 단계에서 현재 가장 빈도수가 작은 두 노드를 선택하는 국소적 최적선택을 반복한다. 이러한 선택을 반복하면 전체 평균 코드 길이가 최소가 되는 최적의 압축 트리를 만들 수 있으므로 탐욕 알고리즘에 해당한다.

# 05 큰 수 만들기

In [14]:
# =========================================
# 큰 수 만들기 (Stack Greedy)
# HTML 시각화 버전
# =========================================

from pyvis.network import Network
import os

# =========================================
# 알고리즘
# =========================================

def solution(number, k):

    stack = []

    initial_k = k

    steps = []

    print(f"입력 숫자: {number}, 제거할 개수: {k}")

    print("-" * 40)

    for i, num in enumerate(number):

        # 제거 과정
        while stack and k > 0 and stack[-1] < num:

            removed = stack.pop()

            k -= 1

            print(
                f"[제거] {removed} < {num} | "
                f"남은 제거 횟수: {k} | "
                f"현재 스택: {''.join(stack)}"
            )

            steps.append({
                "type": "remove",
                "current": num,
                "removed": removed,
                "stack": stack.copy(),
                "k": k
            })

        # 추가
        stack.append(num)

        print(f"[추가] {num} 삽입 | 현재 스택: {''.join(stack)}")

        steps.append({
            "type": "add",
            "current": num,
            "stack": stack.copy(),
            "k": k
        })

    # 남은 제거 처리
    if k > 0:

        print(f"\n[예외] 제거 횟수가 {k}번 남았습니다. 뒤에서 잘라냅니다.")

        stack = stack[:-k]

        steps.append({
            "type": "trim",
            "stack": stack.copy(),
            "k": 0
        })

    result = "".join(stack)

    print("-" * 40)

    print(f"최종 결과: {result}")

    # HTML 생성
    generate_html(number, initial_k, result, steps)

    return result


# =========================================
# HTML 생성
# =========================================

def generate_html(number, initial_k, result, steps):

    html = f"""
<!DOCTYPE html>
<html lang="ko">

<head>

<meta charset="UTF-8">

<title>큰 수 만들기 시각화</title>

<style>

body {{

    margin:0;

    padding:0;

    background:white;

    font-family:Arial;

    overflow-x:hidden;
}}

.container {{

    padding:40px;

}}

.title {{

    font-size:42px;

    font-weight:bold;

    color:#8CC0EB;

    margin-bottom:10px;
}}

.subtitle {{

    color:#BFDDF0;

    font-size:18px;

    margin-bottom:30px;

    font-weight:bold;
}}

.info-box {{

    background:#FFF9D2;

    border:3px solid #8CC0EB;

    border-radius:24px;

    padding:24px;

    width:500px;

    box-shadow:0 10px 30px rgba(140,192,235,0.2);

    margin-bottom:40px;
}}

.info-item {{

    margin-bottom:14px;

    font-size:18px;
}}

.label {{

    color:#8CC0EB;

    font-weight:bold;
}}

.steps {{

    display:flex;

    flex-direction:column;

    gap:28px;
}}

.step-card {{

    background:white;

    border:3px solid #BFDDF0;

    border-radius:28px;

    padding:24px;

    box-shadow:0 8px 24px rgba(140,192,235,0.15);

    transition:0.3s;
}}

.step-card:hover {{

    transform:translateY(-4px);
}}

.step-title {{

    font-size:24px;

    font-weight:bold;

    color:#8CC0EB;

    margin-bottom:18px;
}}

.stack-container {{

    display:flex;

    gap:10px;

    margin-top:16px;

    flex-wrap:wrap;
}}

.stack-box {{

    width:64px;

    height:64px;

    border-radius:18px;

    display:flex;

    align-items:center;

    justify-content:center;

    font-size:28px;

    font-weight:bold;

    background:#8CC0EB;

    color:black;

    border:3px solid #BFDDF0;

    box-shadow:0 4px 10px rgba(0,0,0,0.08);
}}

.remove-box {{

    background:#FFEBCD !important;
}}

.final-box {{

    background:#8CC0EB;

    color:black;

    padding:28px;

    border-radius:30px;

    font-size:32px;

    font-weight:bold;

    width:fit-content;

    margin-top:40px;

    box-shadow:0 10px 30px rgba(140,192,235,0.25);
}}

.badge {{

    display:inline-block;

    padding:8px 16px;

    border-radius:999px;

    font-size:14px;

    font-weight:bold;

    margin-bottom:16px;
}}

.add-badge {{

    background:#8CC0EB;

    color:black;
}}

.remove-badge {{

    background:#FFEBCD;

    color:black;
}}

.trim-badge {{

    background:#FFF9D2;

    color:black;
}}

</style>

</head>

<body>

<div class="container">

<div class="title">
큰 수 만들기
</div>

<div class="subtitle">
Stack + Greedy Algorithm Visualization
</div>

<div class="info-box">

<div class="info-item">
<span class="label">입력 숫자:</span>
{number}
</div>

<div class="info-item">
<span class="label">제거 개수:</span>
{initial_k}
</div>

<div class="info-item">
<span class="label">최종 결과:</span>
{result}
</div>

</div>

<div class="steps">
"""

    # =========================================
    # 단계 추가
    # =========================================

    for idx, step in enumerate(steps, start=1):

        step_type = step["type"]

        if step_type == "add":

            badge = '<div class="badge add-badge">ADD</div>'

            title = f"{idx}. 숫자 {step['current']} 추가"

        elif step_type == "remove":

            badge = '<div class="badge remove-badge">REMOVE</div>'

            title = (
                f"{idx}. {step['removed']} 제거 "
                f"({step['removed']} < {step['current']})"
            )

        else:

            badge = '<div class="badge trim-badge">TRIM</div>'

            title = f"{idx}. 남은 숫자 뒤에서 제거"

        html += f"""
        <div class="step-card">

        {badge}

        <div class="step-title">
        {title}
        </div>

        <div style="
        color:#BFDDF0;
        font-weight:bold;
        margin-bottom:12px;
        ">
        남은 제거 횟수: {step['k']}
        </div>

        <div class="stack-container">
        """

        if len(step["stack"]) == 0:

            html += """
            <div style="
            color:#999;
            font-size:18px;
            ">
            비어 있음
            </div>
            """

        else:

            for num in step["stack"]:

                extra_class = ""

                if step_type == "remove":
                    extra_class = "remove-box"

                html += f"""
                <div class="stack-box {extra_class}">
                {num}
                </div>
                """

        html += """
        </div>
        </div>
        """

    # =========================================
    # 최종 결과
    # =========================================

    html += f"""

    </div>

    <div class="final-box">
    최종 결과: {result}
    </div>

</div>

</body>
</html>
"""

    filename = "largest_number_visualization.html"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"\nHTML 생성 완료: {filename}")


# =========================================
# 테스트 실행
# =========================================

solution("4546476575131400854213565687", 10)

입력 숫자: 4546476575131400854213565687, 제거할 개수: 10
----------------------------------------
[추가] 4 삽입 | 현재 스택: 4
[제거] 4 < 5 | 남은 제거 횟수: 9 | 현재 스택: 
[추가] 5 삽입 | 현재 스택: 5
[추가] 4 삽입 | 현재 스택: 54
[제거] 4 < 6 | 남은 제거 횟수: 8 | 현재 스택: 5
[제거] 5 < 6 | 남은 제거 횟수: 7 | 현재 스택: 
[추가] 6 삽입 | 현재 스택: 6
[추가] 4 삽입 | 현재 스택: 64
[제거] 4 < 7 | 남은 제거 횟수: 6 | 현재 스택: 6
[제거] 6 < 7 | 남은 제거 횟수: 5 | 현재 스택: 
[추가] 7 삽입 | 현재 스택: 7
[추가] 6 삽입 | 현재 스택: 76
[추가] 5 삽입 | 현재 스택: 765
[제거] 5 < 7 | 남은 제거 횟수: 4 | 현재 스택: 76
[제거] 6 < 7 | 남은 제거 횟수: 3 | 현재 스택: 7
[추가] 7 삽입 | 현재 스택: 77
[추가] 5 삽입 | 현재 스택: 775
[추가] 1 삽입 | 현재 스택: 7751
[제거] 1 < 3 | 남은 제거 횟수: 2 | 현재 스택: 775
[추가] 3 삽입 | 현재 스택: 7753
[추가] 1 삽입 | 현재 스택: 77531
[제거] 1 < 4 | 남은 제거 횟수: 1 | 현재 스택: 7753
[제거] 3 < 4 | 남은 제거 횟수: 0 | 현재 스택: 775
[추가] 4 삽입 | 현재 스택: 7754
[추가] 0 삽입 | 현재 스택: 77540
[추가] 0 삽입 | 현재 스택: 775400
[추가] 8 삽입 | 현재 스택: 7754008
[추가] 5 삽입 | 현재 스택: 77540085
[추가] 4 삽입 | 현재 스택: 775400854
[추가] 2 삽입 | 현재 스택: 7754008542
[추가] 1 삽입 | 현재 스택: 77540085421
[추가] 3 삽입 | 현재 스택: 775400854213
[추가

'775400854213565687'

In [12]:
def solution(number, k):
    stack = []
    initial_k = k
    print(f"입력 숫자: {number}, 제거할 개수: {k}")
    print("-" * 40)

    for i, num in enumerate(number):
        # 스택에 값이 있고, 제거할 횟수(k)가 남았으며,
        # 스택의 마지막 값이 현재 숫자보다 작으면 제거 (Pop)
        while stack and k > 0 and stack[-1] < num:
            removed = stack.pop()
            k -= 1
            print(f"[제거] {removed} < {num} | 남은 제거 횟수: {k} | 현재 스택: {''.join(stack)}")

        stack.append(num)
        print(f"[추가] {num} 삽입 | 현재 스택: {''.join(stack)}")

    # 예외 처리: 만약 숫자를 다 돌았는데 k가 남아있다면 (예: "9876", k=2)
    # 뒤에서부터 남은 k만큼 잘라냅니다.
    if k > 0:
        print(f"\n[예외] 제거 횟수가 {k}번 남았습니다. 뒤에서 잘라냅니다.")
        stack = stack[:-k]

    result = "".join(stack)
    print("-" * 40)
    print(f"최종 결과: {result}")
    return result

# 테스트 실행
solution("3498729371312902", 4)

입력 숫자: 3498729371312902, 제거할 개수: 4
----------------------------------------
[추가] 3 삽입 | 현재 스택: 3
[제거] 3 < 4 | 남은 제거 횟수: 3 | 현재 스택: 
[추가] 4 삽입 | 현재 스택: 4
[제거] 4 < 9 | 남은 제거 횟수: 2 | 현재 스택: 
[추가] 9 삽입 | 현재 스택: 9
[추가] 8 삽입 | 현재 스택: 98
[추가] 7 삽입 | 현재 스택: 987
[추가] 2 삽입 | 현재 스택: 9872
[제거] 2 < 9 | 남은 제거 횟수: 1 | 현재 스택: 987
[제거] 7 < 9 | 남은 제거 횟수: 0 | 현재 스택: 98
[추가] 9 삽입 | 현재 스택: 989
[추가] 3 삽입 | 현재 스택: 9893
[추가] 7 삽입 | 현재 스택: 98937
[추가] 1 삽입 | 현재 스택: 989371
[추가] 3 삽입 | 현재 스택: 9893713
[추가] 1 삽입 | 현재 스택: 98937131
[추가] 2 삽입 | 현재 스택: 989371312
[추가] 9 삽입 | 현재 스택: 9893713129
[추가] 0 삽입 | 현재 스택: 98937131290
[추가] 2 삽입 | 현재 스택: 989371312902
----------------------------------------
최종 결과: 989371312902


'989371312902'

왜 탐욕 알고리즘(Greedy Algorithm)인가요?
* 탐욕 알고리즘은 "매 순간, 나중에 미칠 영향을 고려하지 않고 당장 최선의 선택을 하는 방식"을 말합니다.

- 지역적 최적 선택: 본 풀이에서는 숫자를 하나씩 확인할 때마다 "이 숫자가 내 앞에 있는 숫자보다 큰가?"만을 비교합니다. 전체 숫자를 다 훑어보고 조합을 만드는 것이 아니라, 현재 단계에서 가장 큰 자릿수를 만들기 위해 작은 숫자를 즉시 버리는 선택을 하기 때문에 탐욕 알고리즘에 해당합니다.

왜 항상 최적해(가장 큰 수)를 보장하나요?
* 이 문제에서 최적해를 보장하는 이유는 숫자의 '자릿수 특징' 때문입니다.

- 높은 자릿수의 결정력: 숫자의 크기는 가장 왼쪽(높은 자릿수)에 어떤 숫자가 오느냐에 따라 결정됩니다. 뒤에 아무리 큰 숫자들이 나와도, 앞자리가 작으면 전체 숫자의 값은 작아집니다.

- 순차적 우위 확보: 스택을 이용한 방식은 왼쪽부터 차례대로 숫자를 채워나가되, 더 큰 숫자가 나타나면 기존의 작은 숫자를 '즉시' 제거합니다. 이는 가장 높은 자릿수부터 최대한 큰 숫자로 채워나가는 것을 보장합니다.

- 반례 없는 제거: 작은 숫자를 뒤로 미루는 것보다 지금 당장 제거하고 더 큰 숫자를 앞쪽으로 당기는 것이 수치적으로 항상 이득입니다. 따라서 매 순간의 최선의 선택(작은 앞자리 제거)이 모여 결국 전체의 최적해(가장 큰 수)를 만들게 됩니다.